# Assembling the final DeepSeek model

This notebook uses one model for full-sequence training, compressed-cache prefill/decode, hidden-state inspection, and checkpoint restoration.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'model').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch

from src.model.final_deepseek import TinyDeepSeek
from src.model import DeepSeekConfig, DeepSeekMoE, FinalDeepSeekBlock, SwiGLUFFN
from src.training.checkpoint import load_checkpoint, save_checkpoint

torch.manual_seed(22)

In [ ]:
config = DeepSeekConfig(
    vocab_size=256,
    max_seq_len=32,
    dim=64,
    n_layers=3,
    n_dense_layers=1,
    n_heads=4,
    q_lora_rank=16,
    kv_lora_rank=8,
    qk_nope_head_dim=8,
    qk_rope_head_dim=4,
    v_head_dim=8,
    inter_dim=128,
    moe_inter_dim=32,
    n_routed_experts=4,
    n_activated_experts=2,
)
model = TinyDeepSeek(config).eval()
print([type(layer).__name__ for layer in model.layers])
print([type(layer.ffn).__name__ for layer in model.layers])

In [ ]:
tokens = torch.randint(0, config.vocab_size, (2, 8))
logits, hidden = model(tokens, return_hidden=True)
print('input:', tuple(tokens.shape))
print('hidden:', tuple(hidden.shape))
print('logits:', tuple(logits.shape))
assert hidden.shape == (2, 8, config.dim)
assert logits.shape == (2, 8, config.vocab_size)

## Full forward versus cached decode

The last-token logits should agree when the prompt is prefetched and the final token is decoded through the compressed KV caches.

In [ ]:
with torch.no_grad():
    full_logits = model(tokens)
    model.prefill(tokens[:, :-1])
    cached_logits = model.decode_step(tokens[:, -1:])

torch.testing.assert_close(
    cached_logits,
    full_logits[:, -1:],
    rtol=3e-4,
    atol=3e-4,
)
print('cache length:', model.cache_length)
print('full and cached final logits agree')

In [ ]:
for layer in model.layers:
    assert isinstance(layer, FinalDeepSeekBlock)
    assert layer.attention.cache_length == tokens.shape[1]

model.reset_cache()
assert model.cache_length == 0
print('all caches reset')

## Checkpoint round trip

The configuration and router bias buffers are restored with the model weights.

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / 'tiny-final.pt'
    save_checkpoint(path, model, step=22, note='assembly')
    restored, metadata = load_checkpoint(path)

restored_logits = restored(tokens)
torch.testing.assert_close(restored_logits, model(tokens))
print('restored step:', metadata['step'])
print('checkpoint round trip passed')